In [ ]:
import pandas as pd
import re
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Support ticket dataset
data = {
    "text": [
        "cannot log in after password reset",
        "account locked after too many attempts",
        "forgot password link is not working",
        "unable to verify my email address",
        "profile access is blocked after sign in",
        "two factor code never arrives to my phone",
        "session expires immediately after logging in",
        "security question cannot be reset",
        "username recovery email never arrived",
        "password reset email goes to spam",
        "payment was charged twice on my card",
        "refund has not been credited yet",
        "subscription renewal charged the wrong amount",
        "invoice shows an extra billing fee",
        "billing address was rejected during checkout",
        "transaction failed but amount was deducted",
        "card payment keeps getting declined",
        "cancelled subscription still shows a charge",
        "promotional discount did not apply to my order",
        "receipts are missing from my account",
        "app crashes whenever I open the settings page",
        "screen freezes during checkout flow",
        "application keeps restarting on launch",
        "search results are not loading in the app",
        "audio is delayed during video playback",
        "unable to upload profile picture",
        "error message appears when submitting the form",
        "notifications are not coming through",
        "dark mode toggle does nothing",
        "dashboard widgets are blank after update"
    ],
    "label": [
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Account Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Billing Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue",
        "Technical Issue"
    ]
}

df = pd.DataFrame(data)

# Preprocessing
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(preprocess)

# TF-IDF + linear SVM pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('model', LinearSVC(class_weight='balanced'))
])

y = df['label']
texts = df['clean_text'].to_numpy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(pipeline, texts, y, cv=cv)
print(classification_report(y, y_pred, zero_division=0))

pipeline.fit(texts, y)

# Prediction function
def predict_query(query):
    clean_query = preprocess(query)
    return pipeline.predict([clean_query])[0]

# Test
print(predict_query("I cannot login to my account"))
print(predict_query("payment deducted but failed"))

                 precision    recall  f1-score   support

  Account Issue       0.67      0.80      0.73        10
  Billing Issue       0.64      0.70      0.67        10
Technical Issue       0.57      0.40      0.47        10

       accuracy                           0.63        30
      macro avg       0.62      0.63      0.62        30
   weighted avg       0.62      0.63      0.62        30

Account Issue
Billing Issue
